In [19]:
from platform import system

from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic

client = Anthropic()
# 4.7 and later models no longer accept temperature
model = "claude-sonnet-4-5"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature

    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            return block.text

In [37]:
messages = []
add_user_message(
    messages,
    "Generate a very short EventBridge rule as JSON in a ```json code fence."
)
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [38]:
import json

json.loads(text.strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [42]:
# EXERCISE
# Use message prefilling and stop sequences only to get three different commands in a single response
# There shouldn't be any comments or explanation
# Hint: message prefilling isn't limited to just characters like ```

messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments: ```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls\naws ec2 describe-instances\naws iam list-users'